# FineSightBench 数据生成库使用演示

本 Notebook 展示：
1. **数据集一（Perception）** — 细粒度视觉感知评测集的生成与可视化
2. **数据集二（Reasoning）** — 细粒度视觉推理评测集的生成与可视化
3. **底层绘制 API** — 直接使用核心库绘制单个目标
4. **干扰效果演示** — 色盲模拟 & 模糊处理

In [1]:
%matplotlib inline
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 核心绘制 API
from finesightbench.core import (
    create_canvas, COLORS, simulate_cvd, apply_blur,
    draw_letter, draw_animal, draw_block, draw_color_block, draw_shape, draw_dot,
    ANIMAL_TYPES, SHAPE_TYPES, LETTERS,
)
# 数据集生成器
from finesightbench.perception import generate_perception_dataset
from finesightbench.reasoning import generate_reasoning_dataset
# 可视化工具
from finesightbench.visualize import visualize_dataset, visualize_by_task

print("✅ 库加载成功")

✅ 库加载成功


## 1. 底层绘制 API 演示

直接使用 `finesightbench.core` 在画布上绘制不同类型、不同尺寸的目标。

In [2]:
# 展示所有目标类型，尺寸 32px
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

demos = [
    ("字母 (Letter)", lambda img: draw_letter(img, (230, 230), 48, "A", COLORS["red"])),
    ("小动物 (Animal)", lambda img: draw_animal(img, (230, 230), 48, "cat", COLORS["blue"])),
    ("方块 (Block)", lambda img: draw_block(img, (230, 230), 48, COLORS["black"])),
    ("颜色方块 (Color Block)", lambda img: draw_color_block(img, (230, 230), 48, COLORS["green"])),
    ("形状 (Shape - star)", lambda img: draw_shape(img, (225, 225), 56, "star", COLORS["purple"])),
    ("圆点 (Dot)", lambda img: draw_dot(img, (235, 235), 40, COLORS["orange"])),
]

for ax, (title, draw_fn) in zip(axes.flat, demos):
    img = create_canvas(512, 512)
    draw_fn(img)
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis("off")

plt.suptitle("六种目标类型演示 (48px)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 23383 (\N{CJK UNIFIED IDEOGRAPH-5B57}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 27597 (\N{CJK UNIFIED IDEOGRAPH-6BCD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 23567 (\N{CJK UNIFIED IDEOGRAPH-5C0F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 21160 (\N{CJK UNIFIED IDEOGRAPH-52A8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 29289 (\N{CJK UNIFIED IDEOGRAPH-7269}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 26041 (\N{CJK UNIFIED IDEOGRAPH-65B9}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3008047195.py:21: UserWarning: Glyph 22359 (\N{CJK UNIFIED I

In [3]:
# 同一种目标在不同尺寸下的效果 —— 展示"极小目标"的挑战性
sizes = [3, 5, 8, 12, 16, 24, 32, 48]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, sz in zip(axes.flat, sizes):
    img = create_canvas(128, 128)  # 小画布便于观察
    cx, cy = 64 - sz // 2, 64 - sz // 2
    draw_letter(img, (cx, cy), sz, "R", COLORS["red"])
    ax.imshow(img)
    ax.set_title(f"{sz}×{sz} px", fontsize=11)
    ax.axis("off")

plt.suptitle("同一字母 'R' 在 8 个尺度下的渲染效果", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 21516 (\N{CJK UNIFIED IDEOGRAPH-540C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 19968 (\N{CJK UNIFIED IDEOGRAPH-4E00}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 23383 (\N{CJK UNIFIED IDEOGRAPH-5B57}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 27597 (\N{CJK UNIFIED IDEOGRAPH-6BCD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 22312 (\N{CJK UNIFIED IDEOGRAPH-5728}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 20010 (\N{CJK UNIFIED IDEOGRAPH-4E2A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2048151132.py:14: UserWarning: Glyph 23610 (\N{CJK UNIFIED I

## 2. 生成数据集一：FineSight-Perception（细粒度视觉感知评测集）

包含 5 种任务：
- **字母识别** (letter_recognition)
- **小动物识别** (animal_recognition)
- **方块检测** (block_recognition)
- **颜色方块识别** (color_block_recognition)
- **形状识别** (shape_recognition)

每种任务 × 8 个尺度 × N 样本

In [4]:
# 生成 Perception 数据集（每种配置 3 个样本用于演示）
perception_dir = "demo_data/perception"
labels_path = generate_perception_dataset(
    output_dir=perception_dir,
    canvas_size=512,
    num_per_config=3,   # 正式使用时可设为 10~50
    seed=42,
)

[Perception] Generated 120 samples → demo_data/perception


In [5]:
# 查看 labels.json 结构
with open(labels_path) as f:
    data = json.load(f)

print(f"数据集名称: {data['dataset_info']['name']}")
print(f"样本总数: {data['dataset_info']['num_samples']}")
print(f"任务类型: {data['dataset_info']['task_types']}")
print(f"目标尺寸: {data['dataset_info']['target_sizes']}")
print(f"\n--- 示例样本 ---")
print(json.dumps(data["samples"][0], indent=2, ensure_ascii=False))

数据集名称: FineSight-Perception
样本总数: 120
任务类型: ['letter', 'animal', 'block', 'color_block', 'shape']
目标尺寸: [3, 5, 8, 12, 16, 24, 32, 48]

--- 示例样本 ---
{
  "image_id": "perception_letter_3px_00000",
  "image_path": "images/perception_letter_3px_00000.png",
  "dataset": "perception",
  "task_type": "letter_recognition",
  "question": "What letter is displayed in the image?",
  "answer": "U",
  "difficulty": "extreme",
  "metadata": {
    "canvas_size": [
      512,
      512
    ],
    "background_color": "white",
    "background_color_rgb": [
      255,
      255,
      255
    ],
    "targets": [
      {
        "type": "letter",
        "value": "U",
        "size": 3,
        "position": [
          22,
          389
        ],
        "color": "green",
        "color_rgb": [
          0,
          180,
          0
        ]
      }
    ]
  }
}


In [6]:
# 可视化 Perception 数据集总览
visualize_dataset(perception_dir, num_display=25, cols=5, figscale=3.0)
# 展示保存的可视化图
img_vis = Image.open(Path(perception_dir) / "visualization.png")
fig, ax = plt.subplots(figsize=(16, 16))
ax.imshow(img_vis)
ax.axis("off")
ax.set_title("FineSight-Perception 总览", fontsize=14)
plt.show()

[Visualize] Saved → demo_data/perception/visualization.png


/home/snt/projects_lujun/FineSightBench/.venv/lib/python3.11/site-packages/IPython/core/events.py:96: UserWarning: Glyph 24635 (\N{CJK UNIFIED IDEOGRAPH-603B}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/home/snt/projects_lujun/FineSightBench/.venv/lib/python3.11/site-packages/IPython/core/events.py:96: UserWarning: Glyph 35272 (\N{CJK UNIFIED IDEOGRAPH-89C8}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)


In [7]:
# 按任务类型分组可视化
task_vis_paths = visualize_by_task(perception_dir, samples_per_task=8, cols=4)

fig, axes = plt.subplots(1, len(task_vis_paths), figsize=(5 * len(task_vis_paths), 5))
for ax, p in zip(axes, task_vis_paths):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem, fontsize=10)
    ax.axis("off")
plt.suptitle("Perception - 按任务类型分组", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

[Visualize] Saved 5 task-level grids → demo_data/perception/vis_by_task


/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 25353 (\N{CJK UNIFIED IDEOGRAPH-6309}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 20219 (\N{CJK UNIFIED IDEOGRAPH-4EFB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 21153 (\N{CJK UNIFIED IDEOGRAPH-52A1}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 31867 (\N{CJK UNIFIED IDEOGRAPH-7C7B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 22411 (\N{CJK UNIFIED IDEOGRAPH-578B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/531435331.py:10: UserWarning: Glyph 32452 (\N{CJK UNIFIED IDEOGRAP

### 2.1 Perception 逐样本目视检查

每种任务类型选取不同尺度的样本，展示原图 + 红框标注目标位置 + 完整 label 信息。

In [13]:
from PIL import ImageDraw as _IDraw

def show_samples_with_labels(data_dir, samples, title="", cols=4, zoom_pad=30):
    """展示样本图片 + 红框标注目标 + label 信息。"""
    n = len(samples)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 5))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i >= n:
            ax.axis("off")
            continue
        s = samples[i]
        img = Image.open(Path(data_dir) / s["image_path"]).copy()
        draw = _IDraw.Draw(img)
        targets = s["metadata"]["targets"]
        # 绘制红框标注每个目标
        for t in targets:
            x, y = t["position"]
            sz = t["size"]
            pad = max(2, sz // 3)
            draw.rectangle(
                [x - pad, y - pad, x + sz + pad, y + sz + pad],
                outline="red", width=2
            )
        ax.imshow(img)
        # 构建 label 文字
        t0 = targets[0]
        label_lines = [
            f"Task: {s['task_type']}",
            f"Q: {s['question'][:60]}",
            f"A: {s['answer']}",
            f"Size: {t0['size']}px | Diff: {s['difficulty']}",
            f"Color: {t0['color']} | Pos: {t0['position']}",
        ]
        if t0["type"] in ("letter", "animal", "shape"):
            label_lines.insert(3, f"Value: {t0['value']}")
        ax.set_title("\n".join(label_lines), fontsize=7, ha="left",
                     loc="left", family="monospace", pad=4)
        ax.axis("off")

    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

# ── Perception: 每种任务取不同尺度各1个样本 ──
p_data = data  # 已加载的 perception labels
task_types_p = list(set(s["task_type"] for s in p_data["samples"]))
selected_p = []
for task in sorted(task_types_p):
    task_samples = [s for s in p_data["samples"] if s["task_type"] == task]
    # 从 extreme, hard, medium, easy 各取 1 个
    seen_diff = set()
    for s in task_samples:
        if s["difficulty"] not in seen_diff:
            selected_p.append(s)
            seen_diff.add(s["difficulty"])
        if len(seen_diff) == 4:
            break

print(f"选取 {len(selected_p)} 个 Perception 样本进行目视检查")
show_samples_with_labels(perception_dir, selected_p,
                         title="Perception 数据集 — 逐样本检查 (红框=目标位置)", cols=4)

选取 20 个 Perception 样本进行目视检查


/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 25454 (\N{CJK UNIFIED IDEOGRAPH-636E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 38598 (\N{CJK UNIFIED IDEOGRAPH-96C6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 36880 (\N{CJK UNIFIED IDEOGRAPH-9010}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 26679 (\N{CJK UNIFIED IDEOGRAPH-6837}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 26412 (\N{CJK UNIFIED IDEOGRAPH-672C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 26816 (\N{CJK UNIFIED I

In [14]:
# 放大查看极小目标 (extreme/hard 难度) —— 裁剪目标周围区域
def show_zoomed_targets(data_dir, samples, title="", cols=5, crop_size=64):
    """裁剪并放大目标区域，方便检查极小目标。"""
    n = len(samples)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i >= n:
            ax.axis("off")
            continue
        s = samples[i]
        img = Image.open(Path(data_dir) / s["image_path"])
        t = s["metadata"]["targets"][0]
        x, y, sz = t["position"][0], t["position"][1], t["size"]
        cx, cy = x + sz // 2, y + sz // 2
        half = crop_size // 2
        # crop with boundary check
        l = max(0, cx - half)
        u = max(0, cy - half)
        r = min(img.width, cx + half)
        d = min(img.height, cy + half)
        crop = img.crop((l, u, r, d)).resize((200, 200), Image.NEAREST)
        ax.imshow(crop)
        ax.set_title(
            f"{s['task_type']}\n{sz}px [{s['difficulty']}]\n"
            f"A: {s['answer']} | {t['color']}",
            fontsize=8, family="monospace"
        )
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

# 选出 extreme 和 hard 难度的样本 zoom in 检查
small_p = [s for s in p_data["samples"] if s["difficulty"] in ("extreme", "hard")]
# 每种任务取 2 个
zoom_selected = []
for task in sorted(task_types_p):
    task_s = [s for s in small_p if s["task_type"] == task][:2
    ]
    zoom_selected.extend(task_s)

print(f"放大查看 {len(zoom_selected)} 个极小目标 (extreme/hard)")
show_zoomed_targets(perception_dir, zoom_selected,
                    title="Perception — 极小目标放大查看 (Nearest-Neighbor 上采样)", cols=5)

放大查看 10 个极小目标 (extreme/hard)


/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 26497 (\N{CJK UNIFIED IDEOGRAPH-6781}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 23567 (\N{CJK UNIFIED IDEOGRAPH-5C0F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 30446 (\N{CJK UNIFIED IDEOGRAPH-76EE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 26631 (\N{CJK UNIFIED IDEOGRAPH-6807}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 25918 (\N{CJK UNIFIED IDEOGRAPH-653E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 22823 (\N{CJK UNIFIED IDEOGRAPH-5927}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 26597 (\N{CJK UNIFIED I

## 3. 生成数据集二：FineSight-Reasoning（细粒度视觉推理评测集）

包含 6 种任务：
- **比较** (comparison) — 哪个目标更大/更小
- **计数** (counting) — 图中有几个目标
- **空间定位** (spatial) — 目标在哪个象限
- **色盲干扰** (interference_cvd) — 色觉缺陷模拟后识别颜色
- **模糊干扰** (interference_blur) — 模糊后识别颜色
- **链式推理** (chain_reasoning) — 多目标从左到右排序

In [8]:
# 生成 Reasoning 数据集
reasoning_dir = "demo_data/reasoning"
labels_path_r = generate_reasoning_dataset(
    output_dir=reasoning_dir,
    canvas_size=512,
    num_per_config=3,
    seed=42,
)

# 查看统计
with open(labels_path_r) as f:
    data_r = json.load(f)
print(f"数据集名称: {data_r['dataset_info']['name']}")
print(f"样本总数: {data_r['dataset_info']['num_samples']}")
print(f"任务类型: {data_r['dataset_info']['task_types']}")

[Reasoning] Generated 144 samples → demo_data/reasoning
数据集名称: FineSight-Reasoning
样本总数: 144
任务类型: ['comparison', 'counting', 'spatial', 'interference_cvd', 'interference_blur', 'chain_reasoning']


In [9]:
# 查看各任务类型的样本示例
for task in data_r['dataset_info']['task_types']:
    sample = next(s for s in data_r['samples'] if s['task_type'] == task)
    print(f"[{task}]")
    print(f"  Q: {sample['question']}")
    print(f"  A: {sample['answer']}")
    print(f"  难度: {sample['difficulty']}")
    print()

[comparison]
  Q: Which object is larger, the one on the left or the one on the right?
  A: right
  难度: extreme

[counting]
  Q: How many letters are there in the image?
  A: 7
  难度: extreme

[spatial]
  Q: In which quadrant of the image is the object located? Answer: top-left, top-right, bottom-left, or bottom-right.
  A: bottom-left
  难度: extreme

[interference_cvd]
  Q: What is the original color of the block in the image? (The image may have been altered by a color-vision simulation.)
  A: purple
  难度: extreme

[interference_blur]
  Q: What color is the block in the image? (The image may be blurred.)
  A: cyan
  难度: extreme

[chain_reasoning]
  Q: List all objects in the image from left to right. Describe each by its color and identity.
  A: pink Y, cyan C, yellow D, orange T
  难度: extreme



In [10]:
# 按任务类型可视化 Reasoning 数据集
task_vis_paths_r = visualize_by_task(reasoning_dir, samples_per_task=8, cols=4)

# 逐个展示每种推理任务
for p in task_vis_paths_r:
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(Image.open(p))
    ax.set_title(p.stem.replace("_", " ").title(), fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

[Visualize] Saved 6 task-level grids → demo_data/reasoning/vis_by_task


### 3.1 Reasoning 逐样本目视检查

每种推理任务选取不同难度的样本，展示原图 + 红框标注 + 完整 label，方便核实生成质量。

In [15]:
# ── Reasoning: 每种任务取 medium 和 easy 各1个样本 (大一些方便看) ──
task_types_r = sorted(set(s["task_type"] for s in data_r["samples"]))
selected_r = []
for task in task_types_r:
    task_samples = [s for s in data_r["samples"] if s["task_type"] == task]
    seen_diff = set()
    for s in task_samples:
        if s["difficulty"] not in seen_diff:
            selected_r.append(s)
            seen_diff.add(s["difficulty"])
        if len(seen_diff) == 4:
            break

print(f"选取 {len(selected_r)} 个 Reasoning 样本进行目视检查")
show_samples_with_labels(reasoning_dir, selected_r,
                         title="Reasoning 数据集 — 逐样本检查 (红框=目标位置)", cols=4)

选取 24 个 Reasoning 样本进行目视检查


/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 25454 (\N{CJK UNIFIED IDEOGRAPH-636E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 38598 (\N{CJK UNIFIED IDEOGRAPH-96C6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 36880 (\N{CJK UNIFIED IDEOGRAPH-9010}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 26679 (\N{CJK UNIFIED IDEOGRAPH-6837}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 26412 (\N{CJK UNIFIED IDEOGRAPH-672C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/2773535560.py:47: UserWarning: Glyph 26816 (\N{CJK UNIFIED I

In [16]:
# Reasoning: 放大查看 extreme/hard 小目标
small_r = [s for s in data_r["samples"] if s["difficulty"] in ("extreme", "hard")]
zoom_r = []
for task in task_types_r:
    task_s = [s for s in small_r if s["task_type"] == task][:2]
    zoom_r.extend(task_s)

print(f"放大查看 {len(zoom_r)} 个 Reasoning 极小目标")
show_zoomed_targets(reasoning_dir, zoom_r,
                    title="Reasoning — 极小目标放大查看", cols=6)

放大查看 12 个 Reasoning 极小目标


/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 26497 (\N{CJK UNIFIED IDEOGRAPH-6781}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 23567 (\N{CJK UNIFIED IDEOGRAPH-5C0F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 30446 (\N{CJK UNIFIED IDEOGRAPH-76EE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 26631 (\N{CJK UNIFIED IDEOGRAPH-6807}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 25918 (\N{CJK UNIFIED IDEOGRAPH-653E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 22823 (\N{CJK UNIFIED IDEOGRAPH-5927}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/4007890456.py:34: UserWarning: Glyph 26597 (\N{CJK UNIFIED I

## 4. 干扰效果演示

展示色盲模拟（protanopia / deuteranopia / tritanopia）和高斯模糊对图像的影响。

In [11]:
# 色盲模拟演示
base_img = create_canvas(256, 256)
colors_demo = ["red", "green", "blue", "yellow", "orange", "purple"]
for i, c in enumerate(colors_demo):
    x = 20 + (i % 3) * 80
    y = 60 + (i // 3) * 100
    draw_color_block(base_img, (x, y), 50, COLORS[c])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Original", "Protanopia (红色盲)", "Deuteranopia (绿色盲)", "Tritanopia (蓝色盲)"]
images = [base_img] + [simulate_cvd(base_img, t) for t in ["protanopia", "deuteranopia", "tritanopia"]]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle("Color Vision Deficiency Simulation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 模糊效果演示
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
blur_levels = [0, 3, 8, 15]
for ax, r in zip(axes, blur_levels):
    img = base_img if r == 0 else apply_blur(base_img, r)
    ax.imshow(img)
    ax.set_title(f"Blur radius={r}", fontsize=10)
    ax.axis("off")
plt.suptitle("Gaussian Blur Interference", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

/tmp/ipykernel_313430/3656677281.py:18: UserWarning: Glyph 32418 (\N{CJK UNIFIED IDEOGRAPH-7EA2}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3656677281.py:18: UserWarning: Glyph 33394 (\N{CJK UNIFIED IDEOGRAPH-8272}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3656677281.py:18: UserWarning: Glyph 30450 (\N{CJK UNIFIED IDEOGRAPH-76F2}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3656677281.py:18: UserWarning: Glyph 32511 (\N{CJK UNIFIED IDEOGRAPH-7EFF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_313430/3656677281.py:18: UserWarning: Glyph 34013 (\N{CJK UNIFIED IDEOGRAPH-84DD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/home/snt/projects_lujun/FineSightBench/.venv/lib/python3.11/site-packages/IPython/core/events.py:96: UserWarning: Glyph 32418 (\N{CJK UNIFIED IDEOGRAPH-7EA2}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/home/snt/project

## 5. 数据集统计

查看两个数据集中各任务类型和难度等级的样本分布。

In [12]:
from collections import Counter

for name, lp in [("Perception", labels_path), ("Reasoning", labels_path_r)]:
    with open(lp) as f:
        d = json.load(f)
    samples = d["samples"]
    task_counts = Counter(s["task_type"] for s in samples)
    diff_counts = Counter(s["difficulty"] for s in samples)
    
    print(f"{'='*50}")
    print(f"  {name} — {len(samples)} 样本")
    print(f"{'='*50}")
    print(f"  按任务类型:")
    for k, v in sorted(task_counts.items()):
        print(f"    {k:30s} {v:4d}")
    print(f"  按难度等级:")
    for k in ["extreme", "hard", "medium", "easy"]:
        print(f"    {k:30s} {diff_counts.get(k, 0):4d}")
    print()

  Perception — 120 样本
  按任务类型:
    animal_recognition               24
    block_recognition                24
    color_block_recognition          24
    letter_recognition               24
    shape_recognition                24
  按难度等级:
    extreme                          30
    hard                             30
    medium                           30
    easy                             30

  Reasoning — 144 样本
  按任务类型:
    chain_reasoning                  24
    comparison                       24
    counting                         24
    interference_blur                24
    interference_cvd                 24
    spatial                          24
  按难度等级:
    extreme                          36
    hard                             36
    medium                           36
    easy                             36



## 6. 清理 demo 数据

取消下面 cell 的注释即可删除演示生成的数据。

In [ ]:
# import shutil
# shutil.rmtree("demo_data", ignore_errors=True)
# print("已清理 demo_data/")